# Exercise 4: storyboard → clips → measure consistency

## Tasks

1. **Plan:** let the model turn the idea into 4 scenes – lake · boat drifts · Ueli with Bruno · sunset.
2. **Generate:** create one scene with Ueli and Bruno in Flow, using the character sheet.
3. **Check:** rate beanie, jacket and dog against the sheet.

## Setup

Select the **Python 3.11** kernel. The API key comes from your `OPENAI_API_KEY` Codespaces secret.

In [ ]:
import base64
import json
import mimetypes
from pathlib import Path
from typing import Literal

import cv2
import pandas as pd
from openai import OpenAI
from PIL import Image, ImageOps
from pydantic import BaseModel

client = OpenAI(max_retries=5, timeout=60)  # reads OPENAI_API_KEY, retries if the API is busy
MODEL = "gpt-6-luna"

DATA = Path("..") / "data"  # the course images and videos

## Helper functions

Two small functions do the API work in this notebook:

- `image_to_input(path)` turns an image file into text (base64), because the API request is sent as text.
- `ask(prompt, answer_format, images)` sends the prompt and the images, writes a log line and returns the answer as a Python dict.

In [ ]:
def image_to_input(path):
    """Turn an image file into the text format the API expects."""
    image_type = mimetypes.guess_type(path)[0]  # e.g. "image/jpeg"
    image_text = base64.b64encode(path.read_bytes()).decode()  # bytes -> text
    return {
        "type": "input_image",
        "image_url": f"data:{image_type};base64,{image_text}",
        "detail": "low",  # low detail = fewer tokens = cheaper
    }


def ask(prompt, answer_format, images):
    """Send a prompt and a list of images. Returns the answer as a dict."""
    # 1. Put the text and the images into one message.
    content = [{"type": "input_text", "text": prompt}]
    for path in images:
        content.append(image_to_input(path))

    # 2. Send the message. The answer must follow answer_format.
    response = client.responses.parse(
        model=MODEL,
        input=[{"role": "user", "content": content}],
        text_format=answer_format,
        reasoning={"effort": "none"},
        max_output_tokens=500,
        store=False,
    )

    # 3. Log prompt, model and raw answer.
    entry = {"prompt": prompt, "model": MODEL, "answer": response.output_text}
    with open("api_log.jsonl", "a", encoding="utf-8") as log:
        log.write(json.dumps(entry, ensure_ascii=False) + "\n")

    # 4. Return the answer as a Python dict.
    return response.output_parsed.model_dump()

## 1. Plan the storyboard

Plan the scenes first, prompt each scene second. The character descriptions stay **exactly the same** in every prompt.

In [ ]:
CHARACTER = "Ueli, red beanie, olive jacket"
DOG = "Bruno, Bernese, red collar"


class Scene(BaseModel):
    id: int
    shot: str
    flow_prompt: str


class Storyboard(BaseModel):
    scenes: list[Scene]  # a storyboard is a list of scenes

In [ ]:
PROMPT = (
    "Write a storyboard with four short scenes for Google Flow, numbered 1 to 4: "
    "1 a wide mountain lake without characters, 2 a drifting boat, "
    f"3 {CHARACTER} together with the dog {DOG}, 4 sunset. "
    "For shot, give the shot size, e.g. wide or close-up. "
    "Each flow_prompt describes camera, action, light and sound."
)

plan = ask(PROMPT, Storyboard, [])  # no images needed
scenes = pd.DataFrame(plan["scenes"])
scenes

In [ ]:
# The Flow prompt for scene 3, with the fixed descriptions in front.
scene_3 = plan["scenes"][2]["flow_prompt"]
print(f"{CHARACTER}. {DOG}. {scene_3}")

## 2. Generate in Flow

Use **Veo 3.1 Lite**, 8 seconds and **Ingredients to Video**. Add `ueli_sheet.jpg` as ingredient and paste the prompt above. Upload the clip as `my_scene.mp4` next to this notebook – or continue with the course clip.

In [ ]:
sheet = DATA / "reference" / "ueli_sheet.jpg"
display(ImageOps.contain(Image.open(sheet), (400, 300)))

video_path = Path("my_scene.mp4")
if not video_path.exists():
    video_path = DATA / "videos" / "ueli_scene3_talk.mp4"  # fallback: the course clip
print("Using", video_path.name)

## 3. Check consistency

One frame per second of the clip is compared with the character sheet. The lake-only scene has no character, so there is nothing to rate.

In [ ]:
Path("frames").mkdir(exist_ok=True)
video = cv2.VideoCapture(str(video_path))

paths = []
for t in range(8):  # Flow clips are 8 seconds long
    video.set(cv2.CAP_PROP_POS_MSEC, t * 1000)  # jump to second t
    ok, frame = video.read()
    if not ok:  # the video ended earlier
        break
    path = Path("frames") / f"frame_{t}s.jpg"
    cv2.imwrite(str(path), frame)
    paths.append(path)

video.release()
print(len(paths), "frames saved in frames/")

Three test images show a good and two bad results: reference + description, reference only (the man drifts), no reference (new man and dog).

In [ ]:
tests = [
    DATA / "reference" / "ueli_scene1.jpg",
    DATA / "reference" / "ueli_drift.jpg",
    DATA / "reference" / "ueli_noref.jpg",
]
for path in tests:
    print(path.name)
    display(ImageOps.contain(Image.open(path), (300, 200)))

In [ ]:
Rating = Literal["match", "differs", "not visible"]


class Consistency(BaseModel):
    beanie: Rating
    jacket: Rating
    dog: Rating


PROMPT = (
    "Image 1 is the reference. Rate the beanie, jacket and dog in image 2. "
    "Use match for a visible match, differs for a visible change, "
    "and not visible when unclear or out of frame."
)

In [ ]:
rows = []
for path in paths + tests:
    row = ask(PROMPT, Consistency, [sheet, path])  # reference first
    row["file"] = path.name
    rows.append(row)

ratings = pd.DataFrame(rows)
ratings

In [ ]:
# The first rows are the clip frames. Share of "match" per feature:
clip_ratings = ratings.iloc[:len(paths)]
(clip_ratings[["beanie", "jacket", "dog"]] == "match").mean()

**Check yourself:** If every frame were generated as a separate image, the video would flicker. Why – and how do video models avoid it?

### Jupyter notebook --footer info--

In [ ]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')